
# ISMLA Individual Project – Google Colab
## Stepwise, Restart-Safe Workflow: DenseNet121 vs VGG16

This notebook is designed for **Google Colab Free** and saves the result of every major step to **Google Drive**.

### Why this notebook is restart-safe
If Colab disconnects, you do **not** need to start again from the beginning.

Each major step checks whether its saved output already exists:

- If the output exists → the step is skipped.
- If the output does not exist → the step runs and saves its result.
- Model checkpoints are saved to Google Drive.
- Metrics, plots, predictions and tables are saved automatically.

### Workflow
1. Mount Google Drive
2. Configure Kaggle
3. Download dataset
4. Identify the source image folder
5. Resize all images to **224 × 224**
6. Reduce JPEG quality to reduce storage size
7. Build a dataset manifest
8. Perform **stratified 70% / 20% / 10% split**
9. Save split CSV files
10. Prepare TensorFlow datasets
11. Train **Model A – DenseNet121**
12. Save DenseNet121 model, history, plots, metrics and predictions
13. Clear memory
14. Train **Model B – VGG16**
15. Save VGG16 model, history, plots, metrics and predictions
16. Compare both models
17. Generate confusion matrices, ROC curves and sample test predictions
18. Save all paper-ready outputs

> **Important:** Before implementation, the student's **PICO framework, dataset and two model choices should be verified/approved**.

---

## Recommended Colab Free Settings

Use:

**Runtime → Change runtime type → Hardware accelerator → GPU**

If a GPU is unavailable, the notebook will still work on CPU, but training will be slower.

Recommended:
- Image size: **224 × 224**
- Batch size: **16**
- Epochs: **12**
- Early stopping: enabled
- Transfer learning: ImageNet backbone frozen initially


In [ ]:

# ============================================================
# STEP 1: Mount Google Drive and create project folders
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import json
import shutil
import time

# ------------------------------------------------------------
# Keep this exact folder name to reuse your saved Drive dataset and checkpoints.
# ------------------------------------------------------------
PROJECT_NAME = "ISMLA_MobileNetV2_vs_ResNet50V2"

DRIVE_ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME

# Main folders saved permanently in Google Drive
RAW_FOLDER        = DRIVE_ROOT / "01_raw_dataset"
RESIZED_FOLDER    = DRIVE_ROOT / "02_resized_224"
SPLIT_FOLDER      = DRIVE_ROOT / "03_split_dataset"
MODELS_FOLDER     = DRIVE_ROOT / "04_models"
RESULTS_FOLDER    = DRIVE_ROOT / "05_results"
FIGURES_FOLDER    = DRIVE_ROOT / "06_figures"
CHECKPOINT_FOLDER = DRIVE_ROOT / "07_checkpoints"

for folder in [
    RAW_FOLDER,
    RESIZED_FOLDER,
    SPLIT_FOLDER,
    MODELS_FOLDER,
    RESULTS_FOLDER,
    FIGURES_FOLDER,
    CHECKPOINT_FOLDER
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder created at:")
print(DRIVE_ROOT)

print("\nSubfolders:")
for folder in [
    RAW_FOLDER,
    RESIZED_FOLDER,
    SPLIT_FOLDER,
    MODELS_FOLDER,
    RESULTS_FOLDER,
    FIGURES_FOLDER,
    CHECKPOINT_FOLDER
]:
    print("-", folder)


In [ ]:

# ============================================================
# STEP 2: Install / import required packages
# ============================================================

!pip -q install kaggle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)

import tensorflow as tf

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print("GPU available:", gpus)
else:
    print("GPU not available. Training will run on CPU.")


# STEP 3 – Configure Kaggle

On the first run, select your `kaggle.json` file when prompted. The notebook
stores it in your private Google Drive project folder and copies it to Colab's
Kaggle configuration directory.

On later sessions, the notebook reuses the private Drive copy automatically.

**Security:** Never upload `kaggle.json` to GitHub or include it in a public
repository. It contains your Kaggle API credentials.


In [ ]:
# ============================================================
# STEP 3: Configure Kaggle API
# ============================================================

from google.colab import files

DRIVE_KAGGLE_JSON = DRIVE_ROOT / "kaggle.json"
LOCAL_KAGGLE_DIR = Path("/root/.kaggle")
LOCAL_KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_KAGGLE_JSON = LOCAL_KAGGLE_DIR / "kaggle.json"

if DRIVE_KAGGLE_JSON.exists():
    print("Using the Kaggle credentials saved in Google Drive.")
    shutil.copy2(DRIVE_KAGGLE_JSON, LOCAL_KAGGLE_JSON)

else:
    print("First-time setup: please select your kaggle.json file.")
    uploaded = files.upload()

    if "kaggle.json" not in uploaded:
        raise ValueError(
            "Please upload the file named exactly 'kaggle.json'."
        )

    DRIVE_KAGGLE_JSON.write_bytes(uploaded["kaggle.json"])
    shutil.copy2(DRIVE_KAGGLE_JSON, LOCAL_KAGGLE_JSON)

os.chmod(LOCAL_KAGGLE_JSON, 0o600)

print("Kaggle API configured successfully.")



# STEP 4 – Download the Kaggle Dataset

Replace the value below:

```python
KAGGLE_DATASET = "username/dataset-name"
```

with the dataset ID from Kaggle.

Example Kaggle URL:

```text
https://www.kaggle.com/datasets/username/dataset-name
```

Dataset ID:

```text
username/dataset-name
```

The notebook saves the downloaded/extracted dataset in Google Drive, so it does not need to be downloaded again in the next session.


In [ ]:

# ============================================================
# STEP 4: Download dataset from Kaggle - RUNS ONLY ONCE
# ============================================================

# ------------------------------------------------------------
# CHANGE THIS
# ------------------------------------------------------------
KAGGLE_DATASET = "aryashah2k/indian-medicinal-leaves-dataset"
DOWNLOAD_DONE_FILE = CHECKPOINT_FOLDER / "dataset_download_complete.txt"

if DOWNLOAD_DONE_FILE.exists():
    print("Dataset download step already completed.")
    print("Skipping download.")

else:
    print("Downloading dataset from Kaggle...")

    # Download and unzip directly into Google Drive
    !kaggle datasets download -d "$KAGGLE_DATASET" -p "$RAW_FOLDER" --unzip

    # Save checkpoint
    DOWNLOAD_DONE_FILE.write_text(
        f"Dataset downloaded successfully.\nDataset: {KAGGLE_DATASET}\n"
    )

    print("Dataset download completed and saved.")

print("\nRaw dataset folder:")
print(RAW_FOLDER)


In [ ]:

# ============================================================
# STEP 5: Inspect downloaded folder structure
# ============================================================

print("Checking downloaded folder structure...\n")

max_depth = 3
shown_files_per_folder = 3

for root, dirs, files_in_dir in os.walk(RAW_FOLDER):
    root_path = Path(root)
    depth = len(root_path.relative_to(RAW_FOLDER).parts)

    if depth > max_depth:
        dirs[:] = []
        continue

    indent = "  " * depth
    print(f"{indent}{root_path.name}/")

    for filename in files_in_dir[:shown_files_per_folder]:
        print(f"{indent}  {filename}")



# STEP 6 – Set the Source Image Folder

After checking the folder structure above, set `SOURCE_FOLDER` to the folder that **directly contains the class folders**.

Example:

```text
02_raw_dataset/
    cats/
    dogs/
```

Then:

```python
SOURCE_FOLDER = RAW_FOLDER
```

Or if the structure is:

```text
01_raw_dataset/
    train/
        cats/
        dogs/
```

then:

```python
SOURCE_FOLDER = RAW_FOLDER / "train"
```


In [ ]:

# ============================================================
# STEP 6: Select source image folder
# ============================================================

# ------------------------------------------------------------
# CHANGE THIS AFTER CHECKING THE FOLDER STRUCTURE
# ------------------------------------------------------------
SOURCE_FOLDER = RAW_FOLDER / "Indian Medicinal Leaves Image Datasets" / "Medicinal Leaf dataset"

print("Selected SOURCE_FOLDER:")
print(SOURCE_FOLDER)

if not SOURCE_FOLDER.exists():
    raise FileNotFoundError("SOURCE_FOLDER does not exist. Please correct the path.")

class_folders = [p for p in SOURCE_FOLDER.iterdir() if p.is_dir()]

print("\nClass folders detected:")
for p in sorted(class_folders):
    print("-", p.name)

if len(class_folders) < 2:
    print("\nWARNING:")
    print("Fewer than 2 class folders were found.")
    print("Check SOURCE_FOLDER before continuing.")



# STEP 7 – Resize and Compress Images

This step:

- opens each image,
- converts it to RGB,
- resizes it to **224 × 224**,
- saves it as JPEG,
- uses JPEG quality = **80**,
- preserves class folders.

The resized dataset is permanently saved to Google Drive.

If the step was completed earlier, the notebook skips it automatically.


In [ ]:

# ============================================================
# STEP 7: Resize all images to 224x224 and reduce JPEG quality
# ============================================================

from PIL import Image

IMG_SIZE = (224, 224)
JPEG_QUALITY = 80

RESIZE_DONE_FILE = CHECKPOINT_FOLDER / "resize_224_complete.txt"

valid_extensions = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"
}

if RESIZE_DONE_FILE.exists():
    print("Image resizing/compression already completed.")
    print("Skipping this step.")

else:
    # Protect existing data if the completion marker is missing.
    if RESIZED_FOLDER.exists() and any(RESIZED_FOLDER.iterdir()):
        raise RuntimeError(
            "The resized-image folder already contains data, but the "
            "completion marker is missing. To protect your saved dataset, "
            "this cell will not delete or rebuild it automatically. "
            "Verify the folder and marker in Google Drive before continuing."
        )

    RESIZED_FOLDER.mkdir(parents=True, exist_ok=True)

    processed = 0
    skipped = 0

    class_dirs = [p for p in SOURCE_FOLDER.iterdir() if p.is_dir()]

    for class_dir in sorted(class_dirs):

        output_class_dir = RESIZED_FOLDER / class_dir.name
        output_class_dir.mkdir(parents=True, exist_ok=True)

        print("Processing class:", class_dir.name)

        for img_path in class_dir.rglob("*"):

            if not img_path.is_file():
                continue

            if img_path.suffix.lower() not in valid_extensions:
                continue

            try:
                with Image.open(img_path) as img:

                    img = img.convert("RGB")
                    img = img.resize(IMG_SIZE)

                    # Create a unique output filename
                    relative_name = "_".join(
                        img_path.relative_to(class_dir).with_suffix("").parts
                    )

                    output_name = relative_name + ".jpg"
                    output_path = output_class_dir / output_name

                    img.save(
                        output_path,
                        format="JPEG",
                        quality=JPEG_QUALITY,
                        optimize=True
                    )

                    processed += 1

            except Exception as e:
                skipped += 1
                print("Skipped:", img_path.name, "|", e)

    RESIZE_DONE_FILE.write_text(
        f"Resize complete.\nProcessed={processed}\nSkipped={skipped}\n"
    )

    print("\nResize step completed.")
    print("Processed:", processed)
    print("Skipped:", skipped)

print("\nResized image folder:")
print(RESIZED_FOLDER)


In [ ]:
from pathlib import Path

class_counts = {}

for folder in sorted(RESIZED_FOLDER.iterdir()):
    if folder.is_dir():
        count = sum(
            1 for f in folder.iterdir()
            if f.is_file()
            and f.suffix.lower() in [".jpg", ".jpeg", ".png"]
        )
        if count > 0:
            class_counts[folder.name] = count

print("Total classes:", len(class_counts))
print("Total images:", sum(class_counts.values()))
print("\nImages per class:")

for name, count in class_counts.items():
    print(f"{name}: {count}")

In [ ]:
print("Total classes:", len(class_counts))
print("Total images:", sum(class_counts.values()))
print("Lowest-count classes:")
for name, count in sorted(class_counts.items(), key=lambda x: x[1])[:10]:
    print(name, count)

In [ ]:
top_25_classes = sorted(
    class_counts.items(),
    key=lambda item: item[1],
    reverse=True
)[:25]

print("Top 25 classes:")
for name, count in top_25_classes:
    print(f"{name}: {count}")

print("\nTotal images in selected classes:",
      sum(count for name, count in top_25_classes))

In [ ]:

import shutil
from pathlib import Path

SUBSET_FOLDER = DRIVE_ROOT / "03_selected_25_classes"
SUBSET_FOLDER.mkdir(parents=True, exist_ok=True)

for class_name, count in top_25_classes:
    source = RESIZED_FOLDER / class_name
    destination = SUBSET_FOLDER / class_name

    if not destination.exists():
        shutil.copytree(source, destination)

print("Selected classes:", len(list(SUBSET_FOLDER.iterdir())))
print("Subset folder:", SUBSET_FOLDER)
print("Subset creation completed.")


In [ ]:
# ============================================================
# STEP 8: Display sample processed images
# ============================================================
import matplotlib.pyplot as plt
from PIL import Image
import random

# Use your selected 25-class dataset
sample_classes = [name for name, count in top_25_classes[:6]]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, class_name in zip(axes.flat, sample_classes):
    class_folder = SUBSET_FOLDER / class_name
    image_files = [
        p for p in class_folder.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    image_path = random.choice(image_files)

    with Image.open(image_path) as img:
        ax.imshow(img.convert("RGB"))

    ax.set_title(class_name)
    ax.axis("off")

plt.suptitle("Sample Processed Medicinal Leaf Images (224 × 224)")
plt.tight_layout()
plt.show()



# STEP 9 – Build Dataset Manifest

A **manifest** is a CSV file containing:

- image path
- class label

The manifest is saved permanently.

This makes splitting reproducible and easy to inspect.


In [ ]:

# ============================================================
# STEP 9: Create image manifest
# ============================================================

MANIFEST_FILE = RESULTS_FOLDER / "dataset_manifest.csv"

if MANIFEST_FILE.exists():
    print("Manifest already exists.")
    data = pd.read_csv(MANIFEST_FILE)

else:
    image_paths = []
    labels = []

    for class_folder in sorted(
        [p for p in RESIZED_FOLDER.iterdir() if p.is_dir()]
    ):
        for image_path in sorted(class_folder.glob("*.jpg")):

            image_paths.append(str(image_path))
            labels.append(class_folder.name)

    data = pd.DataFrame({
        "image_path": image_paths,
        "label": labels
    })

    data.to_csv(MANIFEST_FILE, index=False)

    print("Manifest created and saved.")

print("\nTotal images:", len(data))
print("\nClass distribution:")
print(data["label"].value_counts())

print("\nSaved:", MANIFEST_FILE)


In [ ]:
print("Classes with fewer than 10 images:")
print(class_counts[class_counts < 10].sort_values())

print("\nTotal images:", int(class_counts.sum()))
print("Total classes:", len(class_counts))


# STEP 10 – Stratified 70% / 20% / 10% Split

The split is performed explicitly using `stratify=`.

- **70% Training**
- **20% Validation**
- **10% Testing**

This helps preserve class proportions across all three sets.

The split CSV files are saved to Google Drive and reused on the next run.


In [ ]:
import pandas as pd

MANIFEST_CSV = RESULTS_FOLDER / "dataset_manifest.csv"
data = pd.read_csv(MANIFEST_CSV)

print("Manifest loaded.")
print("Total images:", len(data))
print("Total classes:", data["label"].nunique())

In [ ]:
selected_class_names = [name for name, count in top_25_classes]

data = data[data["label"].isin(selected_class_names)].reset_index(drop=True)

print("Images for splitting:", len(data))
print("Classes for splitting:", data["label"].nunique())

In [ ]:

# ============================================================
# STEP 10: Stratified 70 / 20 / 10 split
# ============================================================

RANDOM_SEED = 42

TRAIN_CSV = RESULTS_FOLDER / "train_split.csv"
VAL_CSV   = RESULTS_FOLDER / "validation_split.csv"
TEST_CSV  = RESULTS_FOLDER / "test_split.csv"

if TRAIN_CSV.exists() and VAL_CSV.exists() and TEST_CSV.exists():

    print("Split files already exist.")
    print("Loading saved split...")

    train_df = pd.read_csv(TRAIN_CSV)
    val_df   = pd.read_csv(VAL_CSV)
    test_df  = pd.read_csv(TEST_CSV)

else:

    # 70% train, 30% temporary
    train_df, temp_df = train_test_split(
        data,
        test_size=0.30,
        random_state=RANDOM_SEED,
        shuffle=True,
        stratify=data["label"]
    )

    # Temporary 30% becomes:
    # 20% validation + 10% test
    # One-third of temporary set = 10% total
    val_df, test_df = train_test_split(
        temp_df,
        test_size=1/3,
        random_state=RANDOM_SEED,
        shuffle=True,
        stratify=temp_df["label"]
    )

    train_df.to_csv(TRAIN_CSV, index=False)
    val_df.to_csv(VAL_CSV, index=False)
    test_df.to_csv(TEST_CSV, index=False)

    print("Stratified split completed and saved.")

total = len(data)

print("\nFinal split:")
print(
    "Training   :",
    len(train_df),
    f"({len(train_df)/total*100:.2f}%)"
)

print(
    "Validation :",
    len(val_df),
    f"({len(val_df)/total*100:.2f}%)"
)

print(
    "Testing    :",
    len(test_df),
    f"({len(test_df)/total*100:.2f}%)"
)


In [ ]:

# ============================================================
# STEP 11: Verify stratification
# ============================================================

def percentage_table(df, name):
    x = (
        df["label"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .rename(name)
    )
    return x

distribution_table = pd.concat(
    [
        percentage_table(data, "Overall %"),
        percentage_table(train_df, "Train %"),
        percentage_table(val_df, "Validation %"),
        percentage_table(test_df, "Test %")
    ],
    axis=1
)

print(distribution_table)

distribution_file = RESULTS_FOLDER / "class_distribution_percentages.csv"
distribution_table.to_csv(distribution_file)

print("\nSaved:", distribution_file)



# STEP 12 – Copy Split Images into Train / Validation / Test Folders

TensorFlow can load data easily from this structure:

```text
03_split_dataset/
    train/
        class_1/
        class_2/
    val/
        class_1/
        class_2/
    test/
        class_1/
        class_2/
```

This step is also checkpointed.


In [ ]:
print("Training image path:")
print(train_df.iloc[0]["image_path"])

print("\nSplit folder:")
print(SPLIT_FOLDER)

In [ ]:

# ============================================================
# STEP 12: Create split image folders
# ============================================================

SPLIT_DONE_FILE = CHECKPOINT_FOLDER / "split_folders_complete.txt"

if SPLIT_DONE_FILE.exists():
    print("Split folders already created.")
    print("Skipping this step.")

else:
    # Do not delete an existing split dataset if its marker is missing.
    if SPLIT_FOLDER.exists() and any(SPLIT_FOLDER.iterdir()):
        raise RuntimeError(
            "The split-dataset folder already contains data, but the "
            "completion marker is missing. This cell will not delete your "
            "saved split folders. Verify them before rebuilding."
        )

    SPLIT_FOLDER.mkdir(parents=True, exist_ok=True)

    def copy_split(df, split_name):

        print("Creating:", split_name)

        for _, row in df.iterrows():

            source_path = Path(row["image_path"])
            label = row["label"]

            destination_dir = SPLIT_FOLDER / split_name / label
            destination_dir.mkdir(parents=True, exist_ok=True)

            destination_path = destination_dir / source_path.name

            if not destination_path.exists():
                shutil.copy2(source_path, destination_path)

    copy_split(train_df, "train")
    copy_split(val_df, "val")
    copy_split(test_df, "test")

    SPLIT_DONE_FILE.write_text("Split folders created successfully.")

    print("Split folders created and saved.")

print("Saved at:", SPLIT_FOLDER)



# STEP 13 – Copy Split Dataset to Colab Local Storage

Reading thousands of images directly from Google Drive can be slow.

Therefore:

- permanent copy stays in Google Drive,
- temporary working copy is created in `/content/`,
- training uses the faster local copy.

This local copy can disappear when Colab disconnects. That is fine because it can be rebuilt from the permanent Google Drive copy.


In [ ]:

# ============================================================
# STEP 13: Copy split dataset from Drive to local Colab storage
# ============================================================

LOCAL_SPLIT_FOLDER = Path("/content/ismla_split_dataset")

if LOCAL_SPLIT_FOLDER.exists():
    shutil.rmtree(LOCAL_SPLIT_FOLDER)

print("Copying dataset from Google Drive to local Colab storage...")

shutil.copytree(
    SPLIT_FOLDER,
    LOCAL_SPLIT_FOLDER
)

print("Local training copy created:")
print(LOCAL_SPLIT_FOLDER)



# STEP 14 – Load TensorFlow Datasets

The same train/validation/test data are used for both models.

Batch size is kept at **16** to make the notebook safer for free Colab.


In [ ]:

# ============================================================
# STEP 14: Load TensorFlow datasets
# ============================================================

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16

train_ds = tf.keras.utils.image_dataset_from_directory(
    LOCAL_SPLIT_FOLDER / "train",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=RANDOM_SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    LOCAL_SPLIT_FOLDER / "val",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    LOCAL_SPLIT_FOLDER / "test",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)

print("\nClasses:", CLASS_NAMES)
print("Number of classes:", NUM_CLASSES)

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)


In [ ]:

# ============================================================
# STEP 15: Save true test labels once
# ============================================================

Y_TRUE_FILE = RESULTS_FOLDER / "y_true_test.npy"

if Y_TRUE_FILE.exists():
    y_true = np.load(Y_TRUE_FILE)
    print("Loaded saved y_true.")

else:
    y_true = np.concatenate(
        [labels.numpy() for images, labels in test_ds]
    )

    np.save(Y_TRUE_FILE, y_true)
    print("Saved y_true.")

print("Number of test samples:", len(y_true))



# MODEL A – DenseNet121

The model uses:

- ImageNet pretrained weights
- frozen convolutional base
- Global Average Pooling
- Dropout
- new output classifier

The best model checkpoint is saved automatically to Google Drive.

If the final DenseNet121 model already exists, training is skipped.


In [ ]:

# STEP 16: Build DenseNet121
# ============================================================

from tensorflow.keras import layers, models

def build_densenet121(num_classes):

    base_model = tf.keras.applications.DenseNet121(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet"
    )

    base_model.trainable = False

    inputs = layers.Input(shape=(224, 224, 3))

    x = tf.keras.applications.densenet.preprocess_input(inputs)

    x = base_model(
        x,
        training=False
    )

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)

    if num_classes == 2:
        outputs = layers.Dense(
            1,
            activation="sigmoid"
        )(x)

        loss = "binary_crossentropy"

    else:
        outputs = layers.Dense(
            num_classes,
            activation="softmax"
        )(x)

        loss = "sparse_categorical_crossentropy"

    model = models.Model(
        inputs,
        outputs,
        name="DenseNet121_Classifier"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss=loss,
        metrics=["accuracy"]
    )

    return model


In [ ]:

# STEP 17: Train DenseNet121 - restart-safe
# ============================================================

DENSENET_FINAL_MODEL = MODELS_FOLDER / "DenseNet121_final.keras"
DENSENET_BEST_MODEL  = MODELS_FOLDER / "DenseNet121_best.keras"
DENSENET_HISTORY_CSV = RESULTS_FOLDER / "DenseNet121_history.csv"

EPOCHS = 12

if DENSENET_FINAL_MODEL.exists():

    print("Final DenseNet121 model already exists.")
    print("Skipping training.")

else:

    if DENSENET_BEST_MODEL.exists():
        print("Found a saved DenseNet121 checkpoint.")
        print("Resuming from:", DENSENET_BEST_MODEL)

        model_a = tf.keras.models.load_model(DENSENET_BEST_MODEL)

    else:
        print("No earlier DenseNet121 checkpoint found.")
        print("Starting a new DenseNet121 model.")

        model_a = build_densenet121(NUM_CLASSES)

    initial_epoch = 0

    if DENSENET_HISTORY_CSV.exists():
        previous_history = pd.read_csv(DENSENET_HISTORY_CSV)

        if len(previous_history) > 0:
            if "epoch" in previous_history.columns:
                initial_epoch = int(previous_history["epoch"].max()) + 1
            else:
                initial_epoch = len(previous_history)

    print("Starting/resuming from epoch:", initial_epoch)

    callbacks_a = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(DENSENET_BEST_MODEL),
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        ),

        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            verbose=1
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.2,
            patience=2,
            min_lr=1e-6,
            verbose=1
        ),

        tf.keras.callbacks.CSVLogger(
            str(DENSENET_HISTORY_CSV),
            append=True
        )
    ]

    if initial_epoch < EPOCHS:
        history_a = model_a.fit(
            train_ds,
            validation_data=val_ds,
            initial_epoch=initial_epoch,
            epochs=EPOCHS,
            callbacks=callbacks_a
        )
    else:
        print("All planned DenseNet121 epochs were already recorded.")

    model_a.save(DENSENET_FINAL_MODEL)

    print("\nDenseNet121 training completed.")
    print("Saved:", DENSENET_FINAL_MODEL)


In [ ]:
import tensorflow as tf
import os

MODELS_PATH = str(MODELS_FOLDER)

densenet = tf.keras.models.load_model(
    os.path.join(MODELS_PATH, "DenseNet121_final.keras")
)

vgg = tf.keras.models.load_model(
    os.path.join(MODELS_PATH, "VGG16_final.keras")
)

print("DENSENET121")
print("Parameters:", f"{densenet.count_params():,}")
print("Model size:", round(os.path.getsize(
    os.path.join(MODELS_PATH, "DenseNet121_final.keras")
) / (1024 * 1024), 2), "MB")

print("\nVGG16")
print("Parameters:", f"{vgg.count_params():,}")
print("Model size:", round(os.path.getsize(
    os.path.join(MODELS_PATH, "VGG16_final.keras")
) / (1024 * 1024), 2), "MB")

In [ ]:
import tensorflow as tf
import os

TEST_DIR = str(SPLIT_FOLDER / "test")

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

print("Test classes:", test_ds.class_names)
print("Number of classes:", len(test_ds.class_names))

In [ ]:
import time
import numpy as np

# Use the same 16 test images
for images, labels in test_ds.take(1):
    sample_images = images[:16]

def measure_time(model):
    # Warm-up
    for _ in range(5):
        output = model(sample_images, training=False)
        _ = output.numpy()  # Synchronize GPU

    times = []

    # Repeat measurement
    for _ in range(20):
        start = time.perf_counter()
        output = model(sample_images, training=False)
        _ = output.numpy()
        times.append(time.perf_counter() - start)

    return np.median(times) / 16 * 1000

print("DenseNet121:", round(measure_time(densenet), 2), "ms/image")
print("VGG16:", round(measure_time(vgg), 2), "ms/image")

In [ ]:
# ============================================================
# STEP 18: Plot saved DenseNet121 training curves
# ============================================================

densenet_history = pd.read_csv(DENSENET_HISTORY_CSV)

# Remove duplicate epoch rows if training was resumed
if "epoch" in densenet_history.columns:
    densenet_history = (
        densenet_history
        .drop_duplicates(subset="epoch", keep="last")
        .sort_values("epoch")
    )

# Accuracy
plt.figure(figsize=(7, 5))
plt.plot(
    densenet_history["accuracy"],
    label="Training Accuracy"
)
plt.plot(
    densenet_history["val_accuracy"],
    label="Validation Accuracy"
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("DenseNet121 - Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()

densenet_acc_plot = FIGURES_FOLDER / "DenseNet121_accuracy_curve.png"
plt.savefig(densenet_acc_plot, dpi=300, bbox_inches="tight")
plt.show()

# Loss
plt.figure(figsize=(7, 5))
plt.plot(
    densenet_history["loss"],
    label="Training Loss"
)
plt.plot(
    densenet_history["val_loss"],
    label="Validation Loss"
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("DenseNet121 - Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

densenet_loss_plot = FIGURES_FOLDER / "DenseNet121_loss_curve.png"
plt.savefig(densenet_loss_plot, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:")
print(densenet_acc_plot)
print(densenet_loss_plot)


# STEP 19 – Evaluate DenseNet121

The notebook saves:

- predicted class
- predicted probability
- overall metrics
- class-wise classification report
- confusion matrix
- ROC curve

After these results are saved, DenseNet121 can be removed from memory.


In [ ]:
# ============================================================
# STEP 19: Evaluation function
# ============================================================

def evaluate_and_save_model(
    model_path,
    model_name,
    test_ds,
    y_true,
    class_names,
    results_folder,
    figures_folder
):

    metrics_file = results_folder / f"{model_name}_metrics.csv"
    pred_file = results_folder / f"{model_name}_predictions.npz"
    report_file = results_folder / f"{model_name}_classification_report.csv"

    # If results already exist, skip prediction
    if (
        metrics_file.exists()
        and pred_file.exists()
        and report_file.exists()
    ):
        print(model_name, "evaluation already completed.")
        return

    print("Loading model:", model_path)

    model = tf.keras.models.load_model(model_path)

    y_prob = model.predict(test_ds, verbose=1)

    num_classes = len(class_names)

    if num_classes == 2:
        y_prob_for_auc = y_prob.ravel()
        y_pred = (y_prob_for_auc >= 0.5).astype(int)

        roc_auc = roc_auc_score(y_true, y_prob_for_auc)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        np.savez(
            pred_file,
            y_true=y_true,
            y_pred=y_pred,
            y_prob=y_prob_for_auc
        )

    else:
        y_pred = np.argmax(y_prob, axis=1)

        roc_auc = roc_auc_score(
            y_true,
            y_prob,
            multi_class="ovr",
            average="macro"
        )

        precision = precision_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0
        )
        recall = recall_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0
        )
        f1 = f1_score(
            y_true, y_pred,
            average="weighted",
            zero_division=0
        )

        np.savez(
            pred_file,
            y_true=y_true,
            y_pred=y_pred,
            y_prob=y_prob
        )

    accuracy = accuracy_score(y_true, y_pred)

    metrics_df = pd.DataFrame([{
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "ROC-AUC": roc_auc
    }])

    metrics_df.to_csv(metrics_file, index=False)

    report_dict = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(class_names))),
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report_dict).transpose()
    report_df.to_csv(report_file)

    print("\nSaved metrics:")
    display(metrics_df)

    print("\nSaved classification report:")
    display(report_df)

    # Clear model from memory
    del model
    tf.keras.backend.clear_session()

    import gc
    gc.collect()

In [ ]:
# ============================================================
# STEP 21: Clear memory before VGG16
# ============================================================

import gc

try:
    del model_a
except NameError:
    pass

tf.keras.backend.clear_session()
gc.collect()

print("Memory cleared before VGG16.")

In [ ]:
# ============================================================
# STEP 20: Evaluate DenseNet121
# ============================================================

evaluate_and_save_model(
    model_path=DENSENET_BEST_MODEL,
    model_name="DenseNet121",
    test_ds=test_ds,
    y_true=y_true,
    class_names=CLASS_NAMES,
    results_folder=RESULTS_FOLDER,
    figures_folder=FIGURES_FOLDER
)


# MODEL B – VGG16

The same:

- dataset,
- training split,
- validation split,
- test split,
- batch size,
- epoch limit,
- early stopping

are used for a fair comparison.


In [ ]:
# ============================================================
# STEP 22: Build VGG16
# ============================================================

def build_vgg16(num_classes):

    base_model = tf.keras.applications.VGG16(
        input_shape=(224, 224, 3),
        include_top=False,
        weights="imagenet"
    )

    base_model.trainable = False

    inputs = layers.Input(
        shape=(224, 224, 3)
    )

    x = tf.keras.applications.vgg16.preprocess_input(
        inputs
    )

    x = base_model(
        x,
        training=False
    )

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)

    if num_classes == 2:

        outputs = layers.Dense(
            1,
            activation="sigmoid"
        )(x)

        loss = "binary_crossentropy"

    else:

        outputs = layers.Dense(
            num_classes,
            activation="softmax"
        )(x)

        loss = "sparse_categorical_crossentropy"

    model = models.Model(
        inputs,
        outputs,
        name="VGG16_Classifier"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss=loss,
        metrics=["accuracy"]
    )

    return model

In [ ]:
# ============================================================
# STEP 23: Train VGG16 - restart-safe
# ============================================================

VGG_FINAL_MODEL = MODELS_FOLDER / "VGG16_final.keras"
VGG_BEST_MODEL  = MODELS_FOLDER / "VGG16_best.keras"
VGG_HISTORY_CSV = RESULTS_FOLDER / "VGG16_history.csv"

EPOCHS = 12

if VGG_FINAL_MODEL.exists():

    print("Final VGG16 model already exists.")
    print("Skipping training.")

else:

    # Resume from a saved checkpoint if Colab disconnected
    if VGG_BEST_MODEL.exists():
        print("Found a saved VGG16 checkpoint.")
        print("Resuming from:", VGG_BEST_MODEL)

        model_b = tf.keras.models.load_model(VGG_BEST_MODEL)

    else:
        print("No earlier VGG16 checkpoint found.")
        print("Starting a new VGG16 model.")

        model_b = build_vgg16(NUM_CLASSES)

    initial_epoch = 0

    if VGG_HISTORY_CSV.exists():

        previous_history = pd.read_csv(VGG_HISTORY_CSV)

        if len(previous_history) > 0:
            if "epoch" in previous_history.columns:
                initial_epoch = int(previous_history["epoch"].max()) + 1
            else:
                initial_epoch = len(previous_history)

    print("Starting/resuming from epoch:", initial_epoch)

    callbacks_b = [

        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(VGG_BEST_MODEL),
            monitor="val_loss",
            save_best_only=True,
            verbose=1
        ),

        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            verbose=1
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.2,
            patience=2,
            min_lr=1e-6,
            verbose=1
        ),

        tf.keras.callbacks.CSVLogger(
            str(VGG_HISTORY_CSV),
            append=True
        )
    ]

    if initial_epoch < EPOCHS:

        history_b = model_b.fit(
            train_ds,
            validation_data=val_ds,
            initial_epoch=initial_epoch,
            epochs=EPOCHS,
            callbacks=callbacks_b
        )

    else:
        print("All planned VGG16 epochs were already recorded.")

    model_b.save(VGG_FINAL_MODEL)

    print("\nVGG16 training completed.")
    print("Saved:", VGG_FINAL_MODEL)

In [ ]:
# ============================================================
# STEP 24: Plot saved VGG16 training curves
# ============================================================

vgg_history = pd.read_csv(VGG_HISTORY_CSV)

if "epoch" in vgg_history.columns:
    vgg_history = (
        vgg_history
        .drop_duplicates(subset="epoch", keep="last")
        .sort_values("epoch")
    )

# Accuracy
plt.figure(figsize=(7, 5))

plt.plot(
    vgg_history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    vgg_history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("VGG16 - Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.tight_layout()

vgg_acc_plot = FIGURES_FOLDER / "VGG16_accuracy_curve.png"

plt.savefig(
    vgg_acc_plot,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Loss
plt.figure(figsize=(7, 5))

plt.plot(
    vgg_history["loss"],
    label="Training Loss"
)

plt.plot(
    vgg_history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("VGG16 - Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()

vgg_loss_plot = FIGURES_FOLDER / "VGG16_loss_curve.png"

plt.savefig(
    vgg_loss_plot,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:")
print(vgg_acc_plot)
print(vgg_loss_plot)

In [ ]:
# ============================================================
# STEP 25: Evaluate VGG16
# ============================================================

evaluate_and_save_model(
    model_path=VGG_BEST_MODEL,
    model_name="VGG16",
    test_ds=test_ds,
    y_true=y_true,
    class_names=CLASS_NAMES,
    results_folder=RESULTS_FOLDER,
    figures_folder=FIGURES_FOLDER
)

In [ ]:
# ============================================================
# STEP 26: Clear memory after VGG16
# ============================================================

import gc

try:
    del model_b
except NameError:
    pass

tf.keras.backend.clear_session()
gc.collect()

print("Memory cleared after VGG16.")


# STEP 27 – Confusion Matrices

These are generated from the saved predictions.

Therefore, the models do **not** need to be loaded again.


In [ ]:
# ============================================================
# STEP 27: Generate and save confusion matrices
# ============================================================

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def save_confusion_matrix(
    model_name,
    class_names
):
    pred_file = RESULTS_FOLDER / f"{model_name}_predictions.npz"

    saved = np.load(pred_file)

    y_true_saved = saved["y_true"]
    y_pred_saved = saved["y_pred"]

    cm = confusion_matrix(
        y_true_saved,
        y_pred_saved,
        labels=list(range(len(class_names)))
    )

    fig, ax = plt.subplots(figsize=(12, 10))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )

    disp.plot(
        ax=ax,
        cmap="Blues",
        xticks_rotation=90,
        colorbar=False,
        values_format="d"
    )

    plt.title(f"{model_name} - Confusion Matrix")
    plt.tight_layout()

    output_file = FIGURES_FOLDER / f"{model_name}_confusion_matrix.png"

    plt.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print("Saved:", output_file)


save_confusion_matrix(
    "DenseNet121",
    CLASS_NAMES
)

save_confusion_matrix(
    "VGG16",
    CLASS_NAMES
)


# STEP 28 – ROC Curves

For binary classification:
- one ROC curve per model.

For multiclass classification:
- one-vs-rest ROC curves for all classes.


In [ ]:
# ============================================================
# STEP 28: Generate and save ROC curves
# ============================================================

from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

def save_roc_plot(
    model_name,
    class_names
):
    pred_file = RESULTS_FOLDER / f"{model_name}_predictions.npz"

    saved = np.load(pred_file)

    y_true_saved = saved["y_true"]
    y_prob_saved = saved["y_prob"]

    num_classes = len(class_names)

    plt.figure(figsize=(10, 8))

    if num_classes == 2:

        fpr, tpr, _ = roc_curve(
            y_true_saved,
            y_prob_saved
        )

        model_auc = auc(fpr, tpr)

        plt.plot(
            fpr,
            tpr,
            label=f"{model_name} (AUC={model_auc:.3f})"
        )

    else:

        y_true_binary = label_binarize(
            y_true_saved,
            classes=np.arange(num_classes)
        )

        for i, class_name in enumerate(class_names):

            # Skip a class if the test set has no positive examples
            if len(np.unique(y_true_binary[:, i])) < 2:
                continue

            fpr, tpr, _ = roc_curve(
                y_true_binary[:, i],
                y_prob_saved[:, i]
            )

            class_auc = auc(fpr, tpr)

            plt.plot(
                fpr,
                tpr,
                label=f"{class_name} (AUC={class_auc:.3f})"
            )

    plt.plot(
        [0, 1],
        [0, 1],
        "--",
        label="Random Classifier"
    )

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{model_name} - ROC Curves (25 Classes)")
    plt.legend(loc="lower right", fontsize=7)
    plt.grid(True)
    plt.tight_layout()

    output_file = FIGURES_FOLDER / f"{model_name}_ROC_curve.png"

    plt.savefig(
        output_file,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    print("Saved:", output_file)


save_roc_plot("DenseNet121", CLASS_NAMES)
save_roc_plot("VGG16", CLASS_NAMES)


# STEP 29 – Final Performance Comparison

This table uses the saved metrics from both models.

It can be directly used as the basis for the performance table in the research paper.


In [ ]:
# ============================================================
# STEP 29: Final model comparison table
# ============================================================

densenet_metrics = pd.read_csv(
    RESULTS_FOLDER / "DenseNet121_metrics.csv"
)

vgg_metrics = pd.read_csv(
    RESULTS_FOLDER / "VGG16_metrics.csv"
)

comparison_df = pd.concat(
    [
        densenet_metrics,
        vgg_metrics
    ],
    ignore_index=True
)

comparison_file = RESULTS_FOLDER / "final_model_comparison.csv"

comparison_df.to_csv(
    comparison_file,
    index=False
)

print("FINAL MODEL COMPARISON")
display(comparison_df.round(4))

print("\nSaved:")
print(comparison_file)

In [ ]:
# ============================================================
# STEP 30: Save performance comparison bar chart
# ============================================================

metrics_to_plot = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC"
]

plot_df = comparison_df.set_index(
    "Model"
)[metrics_to_plot]

ax = plot_df.T.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.xlabel("Performance Metric")
plt.title("DenseNet121 vs VGG16")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.legend(title="Model")
plt.tight_layout()

comparison_plot_file = FIGURES_FOLDER / "final_performance_comparison.png"

plt.savefig(
    comparison_plot_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:")
print(comparison_plot_file)


# STEP 31 – Test Sample Images

This figure shows:

- actual class,
- DenseNet121 prediction,
- VGG16 prediction.

It uses the previously saved predictions, so the models do not need to be loaded again.


In [ ]:
from pathlib import Path

PROJECT_FOLDER = Path(
    "/content/drive/MyDrive/ISMLA_MobileNetV2_vs_ResNet50V2"
)

RESULTS_FOLDER = PROJECT_FOLDER / "05_results"
FIGURES_FOLDER = PROJECT_FOLDER / "06_figures"
LOCAL_SPLIT_FOLDER = PROJECT_FOLDER / "03_split_dataset"

In [ ]:
import numpy as np
import os

densenet_saved = np.load(
    os.path.join(RESULTS_FOLDER, "DenseNet121_predictions.npz")
)

vgg_saved = np.load(
    os.path.join(RESULTS_FOLDER, "VGG16_predictions.npz")
)

y_true = np.load(
    os.path.join(RESULTS_FOLDER, "y_true_test.npy")
)

print("Actual labels:", y_true.shape)
print("DenseNet predictions:", densenet_saved["y_pred"].shape)
print("VGG16 predictions:", vgg_saved["y_pred"].shape)

In [ ]:
import tensorflow as tf

TEST_DIR = os.path.join(PROJECT_FOLDER, "03_split_dataset", "test")

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

print("Number of test images:", len(test_ds.file_paths))
print("Number of classes:", len(test_ds.class_names))
print("Class names:", test_ds.class_names)

In [ ]:

import numpy as np
import os
import matplotlib.pyplot as plt
from PIL import Image

# Load predictions
densenet_pred = densenet_saved["y_pred"]
vgg_pred = vgg_saved["y_pred"]

# Use the exact test dataset order
test_image_paths = test_ds.file_paths
class_names = test_ds.class_names

# Select one image from each of 12 different classes
selected_indices = []

for class_id in range(len(class_names)):
    matching_indices = np.where(y_true == class_id)[0]

    if len(matching_indices) > 0:
        selected_indices.append(matching_indices[0])

    if len(selected_indices) == 12:
        break

# Generate figure
plt.figure(figsize=(16, 12))

for plot_no, i in enumerate(selected_indices):

    img = Image.open(test_image_paths[i]).convert("RGB")

    actual_label = class_names[int(y_true[i])]
    densenet_label = class_names[int(densenet_pred[i])]
    vgg_label = class_names[int(vgg_pred[i])]

    plt.subplot(3, 4, plot_no + 1)
    plt.imshow(img)

    plt.title(
        f"Actual: {actual_label}\n"
        f"DenseNet121: {densenet_label}\n"
        f"VGG16: {vgg_label}",
        fontsize=9
    )

    plt.axis("off")

plt.tight_layout()

sample_prediction_file = os.path.join(
    FIGURES_FOLDER, "test_sample_predictions.png"
)

plt.savefig(sample_prediction_file, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", sample_prediction_file)
print("Different classes displayed:", len(selected_indices))


In [ ]:
import pandas as pd
import os

densenet_report = pd.read_csv(
    os.path.join(RESULTS_FOLDER, "DenseNet121_classification_report.csv")
)

vgg_report = pd.read_csv(
    os.path.join(RESULTS_FOLDER, "VGG16_classification_report.csv")
)

print("DENSENET121")
display(densenet_report)

print("VGG16")
display(vgg_report)

In [ ]:
import numpy as np
import os

class_names = test_ds.class_names
test_image_paths = test_ds.file_paths

print("DENSENET121 – INCORRECT PREDICTIONS")

wrong_densenet = np.where(densenet_pred != y_true)[0]

for i in wrong_densenet[:15]:
    print(
        "Actual:", class_names[y_true[i]],
        "| Predicted:", class_names[densenet_pred[i]],
        "| Image:", os.path.basename(test_image_paths[i])
    )

print("\nVGG16 – INCORRECT PREDICTIONS")

wrong_vgg = np.where(vgg_pred != y_true)[0]

for i in wrong_vgg[:15]:
    print(
        "Actual:", class_names[y_true[i]],
        "| Predicted:", class_names[vgg_pred[i]],
        "| Image:", os.path.basename(test_image_paths[i])
    )

print("\nTotal incorrect predictions:")
print("DenseNet121:", len(wrong_densenet))
print("VGG16:", len(wrong_vgg))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

fig, axes = plt.subplots(2, 4, figsize=(16, 9))

models = [
    ("DenseNet121", wrong_densenet, densenet_pred),
    ("VGG16", wrong_vgg, vgg_pred)
]

for row, (model_name, wrong_indices, predictions) in enumerate(models):
    for col in range(4):
        axes[row, col].axis("off")

        if col >= len(wrong_indices):
            continue

        idx = wrong_indices[col]
        image = Image.open(test_image_paths[idx]).convert("RGB")

        axes[row, col].imshow(image)
        axes[row, col].set_title(
            f"{model_name}\nActual: {class_names[y_true[idx]]}\n"
            f"Predicted: {class_names[predictions[idx]]}",
            fontsize=9
        )

plt.tight_layout()

error_figure_path = os.path.join(
    FIGURES_FOLDER, "sample_misclassified_images.png"
)

plt.savefig(error_figure_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", error_figure_path)

In [ ]:
import numpy as np
from scipy.stats import binomtest

# Check whether each model predicted correctly
densenet_correct = (densenet_pred == y_true)
vgg_correct = (vgg_pred == y_true)

# Count paired disagreements
b = np.sum(densenet_correct & ~vgg_correct)
c = np.sum(~densenet_correct & vgg_correct)

print("McNemar Test Results")
print("DenseNet121 correct, VGG16 wrong:", b)
print("VGG16 correct, DenseNet121 wrong:", c)

if b + c > 0:
    result = binomtest(
        min(b, c),
        n=b + c,
        p=0.5,
        alternative="two-sided"
    )
    print("Test statistic (min of discordant counts):", min(b, c))
    print("P-value:", result.pvalue)
else:
    print("The models have no discordant predictions; the test is not applicable.")



# STEP 32 – Optional: Generate a List of All Saved Results

This helps the student confirm that all project outputs are available before writing the research paper.


In [ ]:
# ============================================================
# STEP 32: Show all saved project outputs
# ============================================================

print("PROJECT ROOT:")
print(DRIVE_ROOT)

print("\nMODELS:")
for p in sorted(MODELS_FOLDER.glob("*")):
    print("-", p.name)

print("\nRESULT FILES:")
for p in sorted(RESULTS_FOLDER.glob("*")):
    print("-", p.name)

print("\nFIGURES:")
for p in sorted(FIGURES_FOLDER.glob("*")):
    print("-", p.name)

print("\nCHECKPOINTS:")
for p in sorted(CHECKPOINT_FOLDER.glob("*")):
    print("-", p.name)

In [ ]:
print("Project folder exists:", DRIVE_ROOT.exists())
print("Project folder contents:")

if DRIVE_ROOT.exists():
    for item in sorted(DRIVE_ROOT.iterdir()):
        print(item.name)


In [ ]:
RESULTS_FOLDER = DRIVE_ROOT / "05_results"

print("Files inside 05_results:")

for item in sorted(RESULTS_FOLDER.iterdir()):
    print(item.name)

In [ ]:
import pandas as pd

comparison_df = pd.read_csv(
    RESULTS_FOLDER / "final_model_comparison.csv"
)

display(comparison_df.round(4))

In [ ]:
from IPython.display import display, Image

print("DenseNet121 Confusion Matrix")
display(Image(filename=str(FIGURES_FOLDER / "DenseNet121_confusion_matrix.png")))

print("VGG16 Confusion Matrix")
display(Image(filename=str(FIGURES_FOLDER / "VGG16_confusion_matrix.png")))

In [ ]:
import pandas as pd

print("DenseNet121 Classification Report")
display(pd.read_csv(RESULTS_FOLDER / "DenseNet121_classification_report.csv").round(4))

print("VGG16 Classification Report")
display(pd.read_csv(RESULTS_FOLDER / "VGG16_classification_report.csv").round(4))

In [ ]:
from IPython.display import display, Image

print("DenseNet121 Accuracy Curve")
display(Image(filename=str(FIGURES_FOLDER / "DenseNet121_accuracy_curve.png")))

print("VGG16 Accuracy Curve")
display(Image(filename=str(FIGURES_FOLDER / "VGG16_accuracy_curve.png")))

print("DenseNet121 Loss Curve")
display(Image(filename=str(FIGURES_FOLDER / "DenseNet121_loss_curve.png")))

print("VGG16 Loss Curve")
display(Image(filename=str(FIGURES_FOLDER / "VGG16_loss_curve.png")))

In [ ]:
import pandas as pd

for model in ["DenseNet121", "VGG16"]:
    print("\n", model)
    history = pd.read_csv(RESULTS_FOLDER / f"{model}_history.csv")
    display(history.tail())

In [ ]:
for model in ["DenseNet121", "VGG16"]:
    history = pd.read_csv(RESULTS_FOLDER / f"{model}_history.csv")
    print(model, "Total recorded epochs:", len(history))
    print("Best validation accuracy:", round(history["val_accuracy"].max() * 100, 2), "%")
    print("Best validation loss:", round(history["val_loss"].min(), 4))

In [ ]:
FIGURES_FOLDER = DRIVE_ROOT / "06_figures"

print("Figures folder exists:", FIGURES_FOLDER.exists())
print("Folder:", FIGURES_FOLDER)


In [ ]:
import sys
import platform
import psutil
import tensorflow as tf
import numpy as np
import pandas as pd
import sklearn

print("Python:", sys.version)
print("Platform:", platform.platform())
print("CPU:", platform.processor())
print("RAM:", round(psutil.virtual_memory().total / (1024**3), 2), "GB")
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

# How to Resume After a Colab Disconnect

When you return to the project:

1. Mount Google Drive and run the imports/setup cells.
2. Configure Kaggle (the private Drive credential copy is reused automatically).
3. Confirm `SOURCE_FOLDER` points to the folder containing the class folders.
4. Load the saved manifest and split CSV files.
5. Copy the saved split dataset to Colab local storage.
6. Load the TensorFlow datasets.
7. Continue from the first unfinished model or evaluation step.

### Saved project structure

The existing Google Drive project folder name is intentionally retained so
that the notebook can find your previously saved dataset and checkpoints:

```text
ISMLA_MobileNetV2_vs_ResNet50V2/
    01_raw_dataset/
    02_resized_224/
    03_selected_25_classes/
    03_split_dataset/
    04_models/
        DenseNet121_best.keras
        DenseNet121_final.keras
        VGG16_best.keras
        VGG16_final.keras
    05_results/
        dataset_manifest.csv
        train_split.csv
        validation_split.csv
        test_split.csv
        DenseNet121_history.csv
        DenseNet121_metrics.csv
        DenseNet121_classification_report.csv
        DenseNet121_predictions.npz
        VGG16_history.csv
        VGG16_metrics.csv
        VGG16_classification_report.csv
        VGG16_predictions.npz
        final_model_comparison.csv
    06_figures/
    07_checkpoints/
```

**Note:** The old folder name is only a storage-path name. The models implemented
in this notebook are DenseNet121 and VGG16. Their checkpoints are kept under
separate filenames. Do not rename or overwrite the existing Drive folder unless
you intentionally want to create a new project directory.

**GitHub security:** Do not commit `kaggle.json`, datasets, or large model
checkpoint files to the public repository. Keep those in your private Google
Drive. Upload the notebook and selected paper-ready figures/results instead.
